# 🐾 Animal Sound Generator — Colab Training

Trains the VAE + Autoencoder on Google Colab's **T4 GPU (16GB VRAM)** instead of your local GTX 1650 (4GB).

### Before you run:
1. **Upload your dataset** to Google Drive → `MyDrive/animal_sound_generator/data/`
   - OR run the download script below (step 3)
2. **Upload existing checkpoints** (optional) to resume training
3. Runtime → Change runtime type → **T4 GPU**

### Training order (run cells in sequence):
1. Setup (clone repo, install, mount drive)
2. Download data (or use Drive)
3. Autoencoder training (~2-3 hrs on T4)
4. VAE fine-tuning (~2-3 hrs on T4)
5. Diffusion training (~3 hrs on T4)
6. Download models back to your local machine

In [ ]:
# @title 1️⃣ Setup — Clone repo, mount Drive, install deps

import os
from google.colab import drive

# Mount Google Drive (for persistent storage)
drive.mount('/content/drive')

# Clone the repo (if not already)
REPO_DIR = '/content/animal_sound_generator'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/grindydev/animal_sound_generator.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}

# Install dependencies
!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install numpy matplotlib pandas scikit-learn librosa soundfile tqdm
!pip install mlflow optuna  # optional

# Verify GPU
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

# Create model directories
!mkdir -p models/autoencoder_checkpoints/train
!mkdir -p models/vae_checkpoints/train
!mkdir -p models/diffusion_checkpoints
!mkdir -p models/classifier_checkpoints/train

## ⚡ Colab-Optimized Config

The key difference from local: **bigger batch sizes + full base_channels=32** since we have 16GB VRAM.

Edit the configs in your scripts BEFORE training (or use these overrides below):

| Setting | Local (GTX 1650 4GB) | Colab (T4 16GB) |
|---------|---------------------|----------------|
| Autoencoder batch | 2 | **8** |
| Autoencoder base_channels | 16 | **32** (149M params) |
| VAE batch | 1 + accum=2 | **4-8** |
| VAE base_channels | 16 | **32** (223M params) |
| Diffusion batch | 4 | **16** |
| HiFi-GAN batch | 8 | **16** |

In [ ]:
# @title 2️⃣ Load Dataset from Google Drive

import os
import tarfile

LOCAL_DATA = '/content/animal_sound_generator/data'
DRIVE_TAR = '/content/drive/MyDrive/animal_audio.tar.gz'
DRIVE_FOLDER = '/content/drive/MyDrive/animal_sound_generator/data'

if os.path.isdir(os.path.join(LOCAL_DATA, 'animal_audio')):
    print('✅ Data already loaded')
elif os.path.exists(DRIVE_TAR):
    print(f'📂 Extracting {DRIVE_TAR}...')
    os.makedirs(LOCAL_DATA, exist_ok=True)
    with tarfile.open(DRIVE_TAR, 'r:gz') as tar:
        tar.extractall(path=LOCAL_DATA)
    print('✅ Extracted!')
elif os.path.isdir(DRIVE_FOLDER):
    print(f'📂 Copying from Drive folder...')
    !cp -r {DRIVE_FOLDER} {LOCAL_DATA}
    print('✅ Copied!')
else:
    print('⚠️  No data found. Upload animal_audio.tar.gz to Drive root, OR')
    print('   set DOWNLOAD_FROM_SCRATCH=True below to run download_data.py')
    DOWNLOAD_FROM_SCRATCH = True  # @param {type:"boolean"}
    if DOWNLOAD_FROM_SCRATCH:
        !python scripts/download_data.py

# Verify data
!ls data/animal_audio/ 2>/dev/null || echo "❌ No data yet!"

In [ ]:
# @title 2b️⃣ Copy models from Drive (optional — resume training)

DRIVE_MODELS = '/content/drive/MyDrive/animal_sound_generator/models'
LOCAL_MODELS = '/content/animal_sound_generator/models'

if os.path.exists(DRIVE_MODELS):
    print(f"📂 Copying checkpoints from Drive...")
    !cp -r {DRIVE_MODELS}/* {LOCAL_MODELS}/ 2>/dev/null
    !ls -lh {LOCAL_MODELS}/*.pth 2>/dev/null
    print("✅ Models synced!")
else:
    print("ℹ️  No models in Drive — training from scratch")
    os.makedirs(DRIVE_MODELS, exist_ok=True)

In [ ]:
# @title 3️⃣ Train Classifier (prerequisite — already done locally, skip if model exists)

CLASSIFIER_PATH = 'models/best_audio_cnn_train.pth'

if os.path.exists(CLASSIFIER_PATH):
    print(f"✅ Classifier already exists: {CLASSIFIER_PATH}")
    print("   Skipping — needed by VAE finetune for class supervision loss")
else:
    print("⚡ Training classifier from scratch (~5 min on T4)...")
    !python src/train_classifier.py

## 🚀 4️⃣ Train Autoencoder (Step 1 — foundation)

**Before running:** Edit `src/vae/train_ae.py` CONFIG:
```python
"base_channels": 32,  # was 16 — now 149M params

"train": {
    "batch_size": 8,     # was 2 — T4 has 4× VRAM
    ...
}
```

Or use sed to patch inline:

In [ ]:
# Patch autoencoder config for Colab
import re

with open('src/vae/train_ae.py', 'r') as f:
    content = f.read()

# Bump base_channels 16 → 32 (149M params — fits T4 easily)
content = content.replace('"base_channels": 16', '"base_channels": 32')

# Bump batch_size 2 → 8
content = re.sub(
    r'("train":\s*\{[^}]*"batch_size":\s*)2',
    r'\g<1>8',
    content
)

with open('src/vae/train_ae.py', 'w') as f:
    f.write(content)

print("✅ Patched: base_channels=32, batch_size=8")

# Train!
!python src/vae/train_ae.py

In [ ]:
# @title 💾 Save autoencoder to Drive (run after training)

DRIVE_MODELS = '/content/drive/MyDrive/animal_sound_generator/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

# Copy the best model + all checkpoints (for resume)
!cp models/best_autoencoder_train.pth {DRIVE_MODELS}/
!cp -r models/autoencoder_checkpoints {DRIVE_MODELS}/ 2>/dev/null

print(f"✅ Autoencoder saved to Drive: {DRIVE_MODELS}/")
!ls -lh {DRIVE_MODELS}/best_autoencoder_train.pth

## 🎨 5️⃣ Train VAE (Step 2 — adds generation capability)

Edit `src/vae/finetune.py` CONFIG similarly:
```python
"base_channels": 32,    # must match autoencoder

"train": {
    "batch_size": 4,     # was 1
    "gradient_accumulation_steps": 2,  # effective batch = 8
    ...
}
```

In [ ]:
# Patch VAE config for Colab
import re

with open('src/vae/finetune.py', 'r') as f:
    content = f.read()

content = content.replace('"base_channels": 16', '"base_channels": 32')

# batch_size 1 → 4
content = re.sub(
    r'("train":\s*\{[^}]*"batch_size":\s*)1',
    r'\g<1>4',
    content
)

with open('src/vae/finetune.py', 'w') as f:
    f.write(content)

print("✅ Patched: base_channels=32, batch_size=4, grad_accum=2")

# Train!
!python src/vae/finetune.py

In [ ]:
# @title 💾 Save VAE to Drive

DRIVE_MODELS = '/content/drive/MyDrive/animal_sound_generator/models'
os.makedirs(DRIVE_MODELS, exist_ok=True)

!cp models/best_vae_finetune_train.pth {DRIVE_MODELS}/
!cp -r models/vae_checkpoints {DRIVE_MODELS}/ 2>/dev/null

print(f"✅ VAE saved to Drive")
!ls -lh {DRIVE_MODELS}/best_vae_finetune_train.pth

## 🌊 6️⃣ Train Diffusion (Optional — Step 3)

Only needed if VAE output is blurry.

In [ ]:
# Train diffusion (optional)
# Edit src/diffusion/train.py CONFIG: batch_size 4 → 16

import re
with open('src/diffusion/train.py', 'r') as f:
    content = f.read()

content = re.sub(
    r'("train":\s*\{[^}]*"batch_size":\s*)4',
    r'\g<1>16',
    content
)

with open('src/diffusion/train.py', 'w') as f:
    f.write(content)

print("✅ Patched: diffusion batch_size=16")

# !python src/diffusion/train.py  # uncomment to run

## 📦 7️⃣ Download Models Back to Local Machine

After training completes, either:

**Option A:** Download from Google Drive (best for large files):
- Go to https://drive.google.com
- Navigate to `MyDrive/animal_sound_generator/models/`
- Download `best_vae_finetune_train.pth` (891 MB for ch=32) and `best_autoencoder_train.pth`

**Option B:** Copy to Drive then sync locally:
```bash
# On your local machine:
# Install Google Drive desktop app, or use rclone
rclone copy gdrive:animal_sound_generator/models/ ./models/ -P
```

**Option C:** Download directly from Colab (slower, may time out for large files):

In [ ]:
# Compress models for download
!tar -czf /content/animal_sound_models.tar.gz models/*.pth

from google.colab import files
# files.download('/content/animal_sound_models.tar.gz')  # uncomment to download

## ⚡ Quick Test — Generate a sound!

After training, test the pipeline:

In [ ]:
# Generate a dog bark!
!python src/generate.py --label Dog --no-diff --temperature 0.7

# Listen in Colab
from IPython.display import Audio, display
import glob

wavs = sorted(glob.glob('generated_audio/*.wav'))
if wavs:
    display(Audio(wavs[-1], rate=22050))
    print(f"Playing: {wavs[-1]}")
else:
    print("No generated files found — check generate.py output above")